# 11 · Local end-to-end test

Runs the real production pipeline on a **few small books**, locally, via the plain process-pool runner (`configs/pipeline.local.toml`: WebDAV source, local paths). No orchestrator, server, or database.

**Prerequisites** (host tools the engine shells out to):
* `scantailor-deviant-cli` on `PATH`
* `tesseract` with the `slk` language pack
* `recode_pdf` (the `archive-pdf-tools` dependency — in this venv)
* VPN up (WebDAV source). Or stage books to disk and use the filesystem backend (last cell).

> In a notebook the runner is used **sequentially** (`max_parallel=1`) — a process pool can't re-import a notebook kernel. For real parallel runs use the CLI: `python -m evilflowers_books_digitalizer run-corpus`.

## 0 · Preflight — tools + connectivity

In [ ]:
import shutil
from evilflowers_books_digitalizer.config import load_settings
from evilflowers_books_digitalizer.sources.webdav import BookSource

for tool in ('scantailor-deviant-cli', 'tesseract', 'recode_pdf'):
    print(f'{tool:24s}', shutil.which(tool) or 'MISSING ✗')

CONFIG = 'configs/pipeline.local.toml'
SOURCE = 'svf'
book_ids = BookSource(load_settings().sources[SOURCE]).list_books()
print(f'\n{SOURCE}: {len(book_ids)} books reachable over WebDAV')

## 1 · Pick the smallest few books (fast test)

In [ ]:
from tqdm.auto import tqdm
src = BookSource(load_settings().sources[SOURCE])
sized = []
for bid in tqdm(book_ids[:20], desc='sizing', unit='book'):
    try:
        sized.append((bid, src.get_book(bid).n_pages))
    except Exception as exc:  # noqa: BLE001
        print('skip', bid, exc)
sized.sort(key=lambda t: t[1])
BOOK_IDS = [bid for bid, _ in sized[:2]]   # <- edit to choose specific books
print('chosen:', sized[:2])

## 2 · Run them through the pipeline
`run_source` digitizes each book (download → ScanTailor → OCR → MRC → metadata → cover → finalize), appends a resumable JSONL report, and returns the status tally. Re-running skips books whose PDF already exists.

In [ ]:
from evilflowers_books_digitalizer.runner import run_source
result = run_source(SOURCE, book_ids=BOOK_IDS, max_parallel=1, config_path=CONFIG)
result['counts']

In [ ]:
import pandas as pd
pd.DataFrame(result['rows'])[
    ['book_id', 'status', 'n_pages', 'pdf_mb', 'language', 'ocr_chars', 'cover_source', 'minutes']
]

## 3 · Inspect the artifacts (PDF, text, cover)

In [ ]:
from evilflowers_books_digitalizer.runtime import load_runtime
out_dir = load_runtime(CONFIG).output_dir / SOURCE
for p in sorted(out_dir.glob(f'{SOURCE}_*')):
    print(f'{p.stat().st_size/1e6:8.2f} MB  {p.name}')

In [ ]:
from PIL import Image
covers = sorted(out_dir.glob('*.cover.jpg'))
Image.open(covers[0]) if covers else 'no cover yet'

In [ ]:
import pypdfium2 as pdfium
pdfs = sorted(out_dir.glob('*.pdf'))
if pdfs:
    print(pdfs[0].name)
    display(pdfium.PdfDocument(str(pdfs[0]))[0].render(scale=120/72).to_pil())
else:
    print('no pdf produced')

## 4 · The live dashboard (one frame)

In [ ]:
from evilflowers_books_digitalizer import monitor as mon
from rich.console import Console
rt = load_runtime(CONFIG)
Console().print(mon.render(rt, mon._book_totals(rt)))
# headless live version:  python -m evilflowers_books_digitalizer monitor --config configs/pipeline.local.toml

## 5 · (Optional) filesystem backend with manually-staged books
Rehearse the production read path without the mount:
```python
from pathlib import Path
base = Path('.cache/raw-scans/svf/SVF-skeny')
for bid in BOOK_IDS:
    src.stage_book(bid, base / bid / 'stream_pages_tif', progress=True)
```
Then set `[source] backend = "filesystem"`, `root = ".cache/raw-scans"`, `[source.paths] svf = "svf/SVF-skeny"` in `configs/pipeline.local.toml` and re-run cell 2 — now it reads from disk like the VM.

For a parallel / headless run, use the CLI instead:
```bash
python -m evilflowers_books_digitalizer run-source svf --limit 5 --config configs/pipeline.local.toml
python -m evilflowers_books_digitalizer monitor --config configs/pipeline.local.toml
```